In [1]:
from urllib.request import urlopen
import requests
import bs4
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity="all"

In [2]:
url = 'https://jolse.com/category/toners-mists/1019/?page=1'
html = urlopen(url)
bs_obj = bs4.BeautifulSoup(html, 'html.parser')

In [3]:
boxes = bs_obj.select('div.description')
len(boxes)
# 상품정보 22개

22

In [4]:
# 첫 페이지 상품정보 정리
prd_list, price_list, sale_price_list = [[] for _ in range(3)]
for box in boxes:
    strong_tag = box.select_one('strong.name')
    li_tag0, li_tag1 = box.findAll('li')
    
    # 추출 뭐이리 극혐이냐;;;
    prd_list.append(strong_tag.text.split(':')[1])
    price_list.append(li_tag0.text.replace('\nPrice', '').strip())
    sale_price_list.append(li_tag1.text.strip())

first_page_df = pd.DataFrame({
    '품목' : prd_list,
    '정상가' : price_list,
    '할인가' : sale_price_list
})

first_page_df

,품목,정상가,할인가
0,Haruharu WONDER Black Rice Hyaluronic Toner 3...,USD 37.00,USD 33.30
1,SOME BY MI Propolis B5 Glow Barrier Calming T...,USD 22.00,USD 19.80
2,Anua Heartleaf 77% Soothing Toner 250ml,USD 28.00,USD 25.20
3,COSRX AHA/BHA CLARIFYING TREATMENT TONER 150ml,USD 17.25,USD 8.29
4,COSRX Full Fit Propolis Synergy Toner 150ml,USD 28.00,USD 10.59
5,Beauty of Joseon Ginseng Essence Water 150ml,USD 24.55,USD 18.41
6,Haruharu WONDER Black Rice Hyaluronic Toner 1...,USD 22.00,USD 19.80
7,ROUND LAB 1025 Dokdo Toner 200ml,USD 17.00,USD 11.99
8,SOME BY MI AHA BHA PHA 30 Days Miracle Toner ...,USD 24.00,USD 14.29
9,COSRX BHA BLACKHEAD POWER LIQUID 100 ML,USD 20.81,USD 11.39


In [5]:
def get_request_product(url):
    html = urlopen(url)
    bs_obj = bs4.BeautifulSoup(html, 'html.parser')
    return bs_obj

In [6]:
def get_product_info(box):
    strong_tag = box.select_one('strong.name')
    li_tag = box.findAll('li')
    
    name = strong_tag.text.split(':')[1]
    price = li_tag[0].text.replace('\nPrice', '').strip()
    sale_price = li_tag[-1].text.strip()
    return {'품목':name, '정상가':price, '할인가':sale_price}

In [7]:
def get_page_product(url):
    global all_product_df
    bs_obj = get_request_product(url)
    boxes = bs_obj.select('div.description')
    
    # 첫 페이지만 22개, 나머지는 20개
    if url.split('=')[1] != '1':
        boxes = boxes[2:]
        
    for box in boxes:
        df = pd.DataFrame(get_product_info(box), index=range(1,2))
        all_product_df = pd.concat([all_product_df, df], axis=0, ignore_index=True)
    return all_product_df

In [8]:
base_url = 'https://jolse.com/category/toners-mists/1019/?page='

all_product_df = pd.DataFrame({
    '품목' : [],
    '정상가' : [],
    '할인가' : []
})

for i in range(1, 23):
    print('%2d번째 페이지 추출중'%i)
    url = base_url + str(i)
    all_product_df = get_page_product(url)

all_product_df

 1번째 페이지 추출중
 2번째 페이지 추출중
 3번째 페이지 추출중
 4번째 페이지 추출중
 5번째 페이지 추출중
 6번째 페이지 추출중
 7번째 페이지 추출중
 8번째 페이지 추출중
 9번째 페이지 추출중
10번째 페이지 추출중
11번째 페이지 추출중
12번째 페이지 추출중
13번째 페이지 추출중
14번째 페이지 추출중
15번째 페이지 추출중
16번째 페이지 추출중
17번째 페이지 추출중
18번째 페이지 추출중
19번째 페이지 추출중
20번째 페이지 추출중
21번째 페이지 추출중
22번째 페이지 추출중


,품목,정상가,할인가
0,Haruharu WONDER Black Rice Hyaluronic Toner 3...,USD 37.00,USD 33.30
1,SOME BY MI Propolis B5 Glow Barrier Calming T...,USD 22.00,USD 19.80
2,Anua Heartleaf 77% Soothing Toner 250ml,USD 28.00,USD 25.20
3,COSRX AHA/BHA CLARIFYING TREATMENT TONER 150ml,USD 17.25,USD 8.29
4,COSRX Full Fit Propolis Synergy Toner 150ml,USD 28.00,USD 10.59
...,...,...,...
435,isoi Bulgarian Rose Pore Tightening Tonic Ess...,USD 37.00,USD 33.30
436,isoi Bulgarian Rose Blemish Care Tonic Essenc...,USD 37.00,USD 33.30
437,MAY ISLAND 7 Days Secret Centella Cica Toner ...,USD 16.50,USD 14.85
438,secretKey Hyaluron Aqua Soft Toner 500ml,USD 22.00,USD 19.80
